In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, random_split

import pandas as pd
import numpy as np
import time

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

Using device: cuda
GPU: NVIDIA GeForce RTX 4050 Laptop GPU


In [3]:
df = pd.read_csv("Data/train.csv")

y = df["label"].values
X = df.drop("label", axis=1).values

X = X / 255.0
X = X.reshape(-1, 1, 28, 28)

X_tensor = torch.tensor(X, dtype=torch.float32)
X_tensor = F.interpolate(X_tensor, size=(32, 32))

y_tensor = torch.tensor(y, dtype=torch.long)

dataset = TensorDataset(X_tensor, y_tensor)

train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

In [4]:
class InceptionBlock(nn.Module):
    def __init__(self, in_channels, c1, c3_reduce, c3, c5_reduce, c5, pool_proj):
        super().__init__()
        
        self.branch1 = nn.Sequential(
            nn.Conv2d(in_channels, c1, 1),
            nn.ReLU()
        )
        
        self.branch2 = nn.Sequential(
            nn.Conv2d(in_channels, c3_reduce, 1),
            nn.ReLU(),
            nn.Conv2d(c3_reduce, c3, 3, padding=1),
            nn.ReLU()
        )
        
        self.branch3 = nn.Sequential(
            nn.Conv2d(in_channels, c5_reduce, 1),
            nn.ReLU(),
            nn.Conv2d(c5_reduce, c5, 5, padding=2),
            nn.ReLU()
        )
        
        self.branch4 = nn.Sequential(
            nn.MaxPool2d(3, stride=1, padding=1),
            nn.Conv2d(in_channels, pool_proj, 1),
            nn.ReLU()
        )
        
    def forward(self, x):
        return torch.cat([
            self.branch1(x),
            self.branch2(x),
            self.branch3(x),
            self.branch4(x)
        ], dim=1)

In [5]:
class InceptionNet(nn.Module):
    def __init__(self):
        super().__init__()
        
        self.conv1 = nn.Conv2d(1, 64, 3, padding=1)
        
        self.inception1 = InceptionBlock(64, 32, 32, 64, 16, 32, 32)
        self.inception2 = InceptionBlock(160, 64, 64, 128, 32, 64, 64)
        
        self.pool = nn.AdaptiveAvgPool2d((1,1))
        self.fc = nn.Linear(320, 10)
        
    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = self.inception1(x)
        x = self.inception2(x)
        x = self.pool(x)
        x = torch.flatten(x, 1)
        x = self.fc(x)
        return x

In [14]:
class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        
        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, stride=stride, padding=1)
        self.bn1 = nn.BatchNorm2d(out_channels)
        
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, padding=1)
        self.bn2 = nn.BatchNorm2d(out_channels)
        
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 1, stride=stride),
                nn.BatchNorm2d(out_channels)
            )
        else:
            self.shortcut = nn.Identity()
    
    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += self.shortcut(x)
        return F.relu(out)

In [15]:
class ResNet(nn.Module):
    def __init__(self):
        super().__init__()
        
        self.conv = nn.Conv2d(1, 64, 3, padding=1)
        self.bn = nn.BatchNorm2d(64)
        
        self.layer1 = ResidualBlock(64, 64)
        self.layer2 = ResidualBlock(64, 128, stride=2)
        
        self.pool = nn.AdaptiveAvgPool2d((1,1))
        self.fc = nn.Linear(128, 10)
    
    def forward(self, x):
        x = F.relu(self.bn(self.conv(x)))
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.pool(x)
        x = torch.flatten(x, 1)
        x = self.fc(x)
        return x

In [16]:
def train_model(model, epochs=5, batch_size=64, lr=0.001):
    
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size)
    
    model = model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    
    for epoch in range(epochs):
        model.train()
        for xb, yb in train_loader:
            xb = xb.to(device)
            yb = yb.to(device)
            
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()
    
    model.eval()
    correct = 0
    total = 0
    
    with torch.no_grad():
        for xb, yb in val_loader:
            xb = xb.to(device)
            yb = yb.to(device)
            preds = torch.argmax(model(xb), 1)
            correct += (preds == yb).sum().item()
            total += yb.size(0)
    
    return correct / total

In [17]:
acc_inception = train_model(InceptionNet(), epochs=5)
acc_resnet = train_model(ResNet(), epochs=5)

print("InceptionNet Accuracy:", acc_inception)
print("ResNet Accuracy:", acc_resnet)

InceptionNet Accuracy: 0.9227380952380952
ResNet Accuracy: 0.8970238095238096
